# Visualize Terrascope data with leafmap
[Leafmap](https://leafmap.org/) is a Python package that allows users to create interactive maps with minimal coding. It is built on several open-source libraries such as folium and ipyleaflet, which make it easy to generate interactive visualisations. Below is a code snippet that demonstrates how to visualize Terrascope data using leafmap.

## Setup

In [4]:
import leafmap
import leafmap.terrascope as terrascope
import rioxarray
import numpy as np
import os
import getpass

Leafmap appends an extra '/' to the end of the JUPYTERHUB_SERVICE_PREFIX. However, by default, JUPYTERHUB_SERVICE_PREFIX already ends with a '/'. This duplication can cause rendering issues, which can be resolved by **temporarily** removing the trailing '/' from the prefix.

In [ ]:
if os.environ.get('JUPYTERHUB_SERVICE_PREFIX'):
    os.environ['JUPYTERHUB_SERVICE_PREFIX'] = os.environ['JUPYTERHUB_SERVICE_PREFIX'][:-1] if os.environ['JUPYTERHUB_SERVICE_PREFIX'][-1] == "/" else os.environ['JUPYTERHUB_SERVICE_PREFIX']

To access Terrascope data, you need to log in with your Terrascope credentials. The code below prompts you to enter your username and password securely.

In [5]:
os.environ['TERRASCOPE_USERNAME'] = input('Username: ')
os.environ['TERRASCOPE_PASSWORD'] = getpass.getpass('Password: ')
terrascope.login()

Set the bounding box for the area of interest. The coordinates are in the format [west, south, east, north]. In this example, we will use the Brussels area. You can change these coordinates to visualize data from a different location. This bbox will be used throughout the notebook.

In [ ]:
# Brussels area: [west, south, east, north]
bbox = [4.2194, 50.7564, 4.4790, 50.9207]

## Search and display NDVI data


Search for NDVI data in the specified area and time range.

In [ ]:
items = terrascope.search_ndvi(
    bbox=bbox,
    start="2025-05-01",
    end="2025-06-01",
    max_cloud_cover=10,
)

print(f"Found {len(items)} scenes:")
for date in terrascope.get_item_dates(items):
    print(f"  {date}")

Visualize the first NDVI scene on an interactive map.

In [ ]:
center = [(bbox[1] + bbox[3]) / 2, (bbox[0] + bbox[2]) / 2]

m = leafmap.Map(center=center, zoom=14)
m.add_basemap("Esri.WorldImagery")
m.add_raster(
    items[0].assets["NDVI"].href,
    layer_name=f"NDVI {items[0].datetime.date()}",
    colormap="RdYlGn",
    vmin=0,
    vmax=250,  # scale = 0.004, offset = -0.08
)
m.add_colormap(cmap="RdYlGn", label="NDVI", vmin=-0.08, vmax=0.92)
m

Add a time slider to explore the NDVI data over time.

In [ ]:
layers = terrascope.create_time_layers(items)

m = leafmap.Map(center=center, zoom=14)
m.add_time_slider(layers, time_interval=1)
m

## Data analysis

In [ ]:
# Clean up stale tile servers before analysis
terrascope.cleanup_tile_servers()

first_item = items[0]
print(f"Analyzing: {first_item.datetime.date()}")

with rioxarray.open_rasterio(first_item.assets["NDVI"].href, mask_and_scale=True) as ds:
    clipped = ds.rio.clip_box(*bbox, crs="EPSG:4326")
    data = clipped.sel(band=1).values

    print(f"\nNDVI Statistics:")
    print(f"  Min: {np.nanmin(data):.2f}")
    print(f"  Max: {np.nanmax(data):.2f}")
    print(f"  Mean: {np.nanmean(data):.2f}")

## Explore data
Discover other collections available in Terrascope and visualize the ESA WorldCover land cover map.

In [ ]:
collections = terrascope.list_collections()
print(f"Available collections ({len(collections)}):")
for c in sorted(collections):
    print(f"  {c}")

In [ ]:
items = terrascope.search(
    collection="esa-worldcover-map-10m-2021-v2",
    bbox=bbox,
)

In [ ]:
m = leafmap.Map(center=center, zoom=14)
m.add_basemap("Esri.WorldImagery")
m.add_raster(
    items[0].assets["Map"].href,
    layer_name="Land cover",
)
m.add_legend(
    title="Land Cover Type",
    builtin_legend="ESA_WorldCover",
)
m

## Cleanup
When you are done with your render, add the '/' back at the end of the prefix and logout of Terrascope.

In [ ]:
if os.environ.get('JUPYTERHUB_SERVICE_PREFIX'):
    os.environ['JUPYTERHUB_SERVICE_PREFIX'] = os.environ['JUPYTERHUB_SERVICE_PREFIX'] if os.environ['JUPYTERHUB_SERVICE_PREFIX'][-1] == "/" else f"{os.environ['JUPYTERHUB_SERVICE_PREFIX']}/"

terrascope.logout()